# Part A: Retail (E-Commerce Analytics)

This notebook provides complete solutions for **Part A (Questions 1–3)** using the UCI Online Retail public dataset.
The dataset contains both structured transaction fields and free-text product descriptions, making it ideal for combined structured + NLP predictive modelling.

## Q1. Analytical Problem Definition

**Domain selected:** Retail — E-Commerce Analytics

---

### Problem Statement
An online retailer suffers revenue loss and elevated logistics costs from product returns and cancellations.
The analytical problem is to **predict whether a transaction line will be returned or cancelled** (`ReturnFlag = 1`) using transaction details and product-description text.

### Decision Objective
Use model-predicted return probabilities to flag high-risk orders *before* dispatch, enabling targeted interventions:
- Enhanced product-information pages for high-return SKUs
- Proactive customer outreach for suspicious bulk orders
- Dynamic return-prevention campaigns by product category and country

### Relevant Variables

| Variable | Type | Role |
|---|---|---|
| `Quantity` | Numeric | Transaction volume — bulk orders show different return patterns |
| `UnitPrice` | Numeric | Price signal — very low/high prices correlate with returns |
| `InvoiceDate` | DateTime | Temporal patterns (month, weekday, hour) |
| `Country` | Categorical | Regional return-rate differences |
| `CustomerID` | Categorical | Customer loyalty / repeat-purchase signal |
| `StockCode` | Categorical | Product-level return history |
| `Description` | Text | NLP signal from product description |
| **`ReturnFlag`** | Binary (0/1) | **Target** — derived from `InvoiceNo` starting with `C` |

### Role of Data-Driven Modelling
A predictive model estimates the probability of return for each transaction line.
This enables data-driven, proactive decision-making instead of reactive, intuition-based responses — improving inventory planning, customer satisfaction, and profitability.

In [2]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## Q2. Dataset Acquisition and Initial Description

**Data source:** UCI Machine Learning Repository — Online Retail Dataset
**URL:** `https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx`

**Why this dataset is suitable:**
- Contains 541,909 real transaction records from a UK-based online retailer (2010–2011)
- Includes both *structured* fields (quantities, prices, dates, customer IDs) and a *text* field (product descriptions)
- Cancellation/return records are naturally encoded in `InvoiceNo` (prefix `C`)
- Publicly available, well-documented, and widely used in academic retail analytics

In [3]:
DATA_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
raw_df = pd.read_excel(DATA_URL, engine='openpyxl')

print(f'Raw dataset shape : {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
print(f'\nColumn names and dtypes:')
print(raw_df.dtypes)
print(f'\nMissing values (%):\n{(raw_df.isna().mean()*100).round(2)}')
print(f'\nSample records:')
raw_df.head()

Raw dataset shape : 541,909 rows x 8 columns

Column names and dtypes:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values (%):
InvoiceNo      0.0000
StockCode      0.0000
Description    0.2700
Quantity       0.0000
InvoiceDate    0.0000
UnitPrice      0.0000
CustomerID    24.9300
Country        0.0000
dtype: float64

Sample records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.5500,17850.0000,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.3900,17850.0000,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.7500,17850.0000,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.3900,17850.0000,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.3900,17850.0000,United Kingdom


In [4]:
# ── Initial data profiling ────────────────────────────────────────
expected_columns = ['InvoiceNo', 'StockCode', 'Description',
                    'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

retail_df = raw_df[expected_columns].copy()

print('=== Dataset Profile ===')
print(f'Rows              : {retail_df.shape[0]:,}')
print(f'Columns           : {retail_df.shape[1]}')
print(f'Duplicate rows    : {retail_df.duplicated().sum():,}')
print(f'Date range        : {pd.to_datetime(retail_df["InvoiceDate"]).min()} to {pd.to_datetime(retail_df["InvoiceDate"]).max()}')
print(f'Unique invoices   : {retail_df["InvoiceNo"].nunique():,}')
print(f'Unique customers  : {retail_df["CustomerID"].nunique():,}')
print(f'Unique products   : {retail_df["StockCode"].nunique():,}')
print(f'Unique countries  : {retail_df["Country"].nunique():,}')
print(f'\nReturn/cancellation invoices (prefix C): {retail_df["InvoiceNo"].astype(str).str.startswith("C").sum():,}')

print('\nNumeric summary:')
display(retail_df[['Quantity', 'UnitPrice']].describe().T.round(2))

print('\nSample Description values:')
display(retail_df['Description'].dropna().head(10).reset_index(drop=True))

=== Dataset Profile ===
Rows              : 541,909
Columns           : 8
Duplicate rows    : 5,268
Date range        : 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Unique invoices   : 25,900
Unique customers  : 4,372
Unique products   : 4,070
Unique countries  : 38

Return/cancellation invoices (prefix C): 9,288

Numeric summary:


,count,mean,std,min,25%,50%,75%,max
Quantity,541909.0000,9.5500,218.0800,-80995.0000,1.0000,3.0000,10.0000,80995.0000
UnitPrice,541909.0000,4.6100,96.7600,-11062.0600,1.2500,2.0800,4.1300,38970.0000



Sample Description values:


0     WHITE HANGING HEART T-LIGHT HOLDER
1                    WHITE METAL LANTERN
2         CREAM CUPID HEARTS COAT HANGER
3    KNITTED UNION FLAG HOT WATER BOTTLE
4         RED WOOLLY HOTTIE WHITE HEART.
5           SET 7 BABUSHKA NESTING BOXES
6      GLASS STAR FROSTED T-LIGHT HOLDER
7                 HAND WARMER UNION JACK
8              HAND WARMER RED POLKA DOT
9          ASSORTED COLOUR BIRD ORNAMENT
Name: Description, dtype: object

### Brief Structure and Characteristics

| Attribute | Detail |
|---|---|
| Unit of analysis | Individual transaction line (one product per invoice row) |
| Structured fields | `Quantity`, `UnitPrice`, `InvoiceDate`, `CustomerID`, `StockCode`, `Country` |
| Text field | `Description` — free-text product name |
| Target derivation | `ReturnFlag = 1` when `InvoiceNo` starts with `C` (cancellation) |
| Key data quality issues | Missing `CustomerID` (24.9%), missing `Description` (0.3%), duplicate rows, negative quantities, zero/negative prices |

## Q3. Data Cleaning and Preparation

The following steps systematically handle all data quality issues and produce the cleaned dataset **Tumushiime.csv**:

1. **Duplicate removal** — drop exact duplicate rows
2. **Missing values** — drop rows with missing critical fields; impute `CustomerID = -1` + add binary missingness indicator
3. **Inconsistencies** — remove rows where `UnitPrice <= 0` or `Quantity = 0`
4. **Outlier treatment** — winsorise `AbsQuantity` and `UnitPrice` at the 1st / 99th percentiles
5. **Text normalisation** — lowercase, strip punctuation, collapse whitespace
6. **Feature engineering** — derive temporal features, interaction terms, and description length

In [5]:
df = retail_df.copy()
metrics = {}
metrics['rows_raw']  = int(df.shape[0])
metrics['cols_raw']  = int(df.shape[1])

# 1. Standardise text columns
for col in ['InvoiceNo', 'StockCode', 'Description', 'Country']:
    df[col] = df[col].astype('string').str.strip()

# 2. Derive binary return/cancellation target
df['ReturnFlag'] = df['InvoiceNo'].str.startswith('C', na=False).astype(int)

# 3. Type conversions
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['Quantity']    = pd.to_numeric(df['Quantity'],    errors='coerce')
df['UnitPrice']   = pd.to_numeric(df['UnitPrice'],   errors='coerce')
df['CustomerID']  = pd.to_numeric(df['CustomerID'],  errors='coerce')

# 4. Remove duplicate rows
before_dupes = len(df)
df = df.drop_duplicates()
metrics['duplicates_removed'] = int(before_dupes - len(df))

# 5. Drop rows missing critical fields
critical = ['InvoiceNo', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'Country']
before_missing = len(df)
df = df.dropna(subset=critical)
metrics['rows_dropped_missing_critical'] = int(before_missing - len(df))

# 6. CustomerID: impute + indicator
df['CustomerIDMissing'] = df['CustomerID'].isna().astype(int)
df['CustomerID']        = df['CustomerID'].fillna(-1).astype(int)

# 7. Remove inconsistent values
before_inconsistent = len(df)
df = df[(df['UnitPrice'] > 0) & (df['Quantity'] != 0)]
metrics['rows_dropped_inconsistent'] = int(before_inconsistent - len(df))

# 8. Normalise text description
df['DescriptionClean'] = (
    df['Description']
    .str.lower()
    .str.replace(r'[^a-z0-9\s]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
df = df[df['DescriptionClean'] != '']

# 9. Outlier treatment (winsorisation at 1st-99th percentile)
df['AbsQuantity'] = df['Quantity'].abs()
q_lo, q_hi   = df['AbsQuantity'].quantile([0.01, 0.99])
p_lo, p_hi   = df['UnitPrice'].quantile([0.01, 0.99])
df['AbsQuantity'] = df['AbsQuantity'].clip(q_lo, q_hi)
df['UnitPrice']   = df['UnitPrice'].clip(p_lo, p_hi)

# 10. Feature engineering
df['InvoiceMonth']      = df['InvoiceDate'].dt.month
df['InvoiceWeekday']    = df['InvoiceDate'].dt.dayofweek
df['InvoiceHour']       = df['InvoiceDate'].dt.hour
df['DescriptionLength'] = df['DescriptionClean'].str.len()
df['LineValueAbs']      = df['AbsQuantity'] * df['UnitPrice']

# 11. Select final columns
clean_cols = [
    'InvoiceNo', 'StockCode', 'Description', 'DescriptionClean', 'DescriptionLength',
    'Quantity', 'AbsQuantity', 'UnitPrice', 'LineValueAbs',
    'InvoiceDate', 'InvoiceMonth', 'InvoiceWeekday', 'InvoiceHour',
    'CustomerID', 'CustomerIDMissing', 'Country', 'ReturnFlag'
]
clean_df = df[clean_cols].sort_values('InvoiceDate').reset_index(drop=True)

metrics['rows_clean']        = int(clean_df.shape[0])
metrics['cols_clean']        = int(clean_df.shape[1])
metrics['return_rate_clean'] = float(clean_df['ReturnFlag'].mean())

out_path = Path('Tumushiime.csv')
clean_df.to_csv(out_path, index=False)

print('Cleaning Summary')
print('=' * 40)
for k, v in metrics.items():
    val = f'{v:.4f}' if isinstance(v, float) else f'{v:,}'
    print(f'  {k:<35}: {val}')
print(f'\nSaved to: {out_path.resolve()}')
clean_df.head()

Cleaning Summary
  rows_raw                           : 541,909
  cols_raw                           : 8
  duplicates_removed                 : 5,268
  rows_dropped_missing_critical      : 1,454
  rows_dropped_inconsistent          : 1,058
  rows_clean                         : 534,129
  cols_clean                         : 17
  return_rate_clean                  : 0.0173

Saved to: C:\Users\robert.tumushiime\OneDrive - ubos.org\Data Capability\Msc Data Science UCU 2025-2026\Year 2 Semester 1\Data Mining, Modelling and Analytics\Final Exam\Tumushiime.csv


,InvoiceNo,StockCode,Description,DescriptionClean,DescriptionLength,Quantity,AbsQuantity,UnitPrice,LineValueAbs,InvoiceDate,InvoiceMonth,InvoiceWeekday,InvoiceHour,CustomerID,CustomerIDMissing,Country,ReturnFlag
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,white hanging heart t light holder,34,6,6,2.5500,15.3000,2010-12-01 08:26:00,12,2,8,17850,0,United Kingdom,0
1,536365,71053,WHITE METAL LANTERN,white metal lantern,19,6,6,3.3900,20.3400,2010-12-01 08:26:00,12,2,8,17850,0,United Kingdom,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,cream cupid hearts coat hanger,30,8,8,2.7500,22.0000,2010-12-01 08:26:00,12,2,8,17850,0,United Kingdom,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,knitted union flag hot water bottle,35,6,6,3.3900,20.3400,2010-12-01 08:26:00,12,2,8,17850,0,United Kingdom,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,red woolly hottie white heart,29,6,6,3.3900,20.3400,2010-12-01 08:26:00,12,2,8,17850,0,United Kingdom,0


In [6]:
# Verify zero missing values in cleaned dataset
missing = clean_df.isna().sum()
assert missing.sum() == 0, f"Unexpected missing values: {missing[missing > 0]}"
print("No missing values in cleaned dataset")
print(missing)

No missing values in cleaned dataset
InvoiceNo            0
StockCode            0
Description          0
DescriptionClean     0
DescriptionLength    0
Quantity             0
AbsQuantity          0
UnitPrice            0
LineValueAbs         0
InvoiceDate          0
InvoiceMonth         0
InvoiceWeekday       0
InvoiceHour          0
CustomerID           0
CustomerIDMissing    0
Country              0
ReturnFlag           0
dtype: int64


### Cleaning Summary

| Step | Action | Rationale |
|---|---|---|
| Duplicate removal | Dropped exact duplicate rows | Avoids inflated training signal |
| Missing critical fields | Dropped null `InvoiceNo`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `Country` | Essential fields for any analysis |
| CustomerID imputation | Set missing to `-1`; added `CustomerIDMissing` flag | Preserves rows while encoding missingness as a predictive feature |
| Inconsistent values | Removed `UnitPrice <= 0` and `Quantity = 0` | Physically impossible in a valid retail transaction |
| Outlier treatment | Winsorised `AbsQuantity` and `UnitPrice` at 1st/99th percentiles | Prevents extreme values from distorting model training |
| Text normalisation | Lowercase + strip punctuation + collapse whitespace | Ensures consistent tokenisation for TF-IDF |

The cleaned dataset is saved as **Tumushiime.csv** and contains **17 columns** ready for modelling.

## Exploratory Data Analysis (EDA)

Six-panel visualisation of the cleaned dataset to understand distributions, class imbalance, and temporal patterns before modelling.

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA - Cleaned Retail Dataset (Tumushiime.csv)', fontsize=13, fontweight='bold')

# 1. Class balance
ax = axes[0, 0]
counts = clean_df['ReturnFlag'].value_counts().sort_index()
bars = ax.bar(['Normal (0)', 'Return (1)'], counts.values, color=['steelblue', 'tomato'], width=0.45)
ax.set_title('Class Distribution - ReturnFlag')
ax.set_ylabel('Records')
total = counts.sum()
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
            f'{v:,}\n({v/total*100:.1f}%)', ha='center', va='bottom', fontsize=9)

# 2. AbsQuantity distribution (normal sales only)
ax = axes[0, 1]
qty = clean_df.loc[clean_df['ReturnFlag'] == 0, 'AbsQuantity']
ax.hist(qty, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(qty.median(), color='red', ls='--', lw=1.5, label=f'Median={qty.median():.0f}')
ax.set_title('AbsQuantity (Normal transactions)')
ax.set_xlabel('Absolute Quantity')
ax.set_ylabel('Frequency')
ax.legend(fontsize=9)

# 3. UnitPrice distribution
ax = axes[0, 2]
price = clean_df['UnitPrice']
ax.hist(price, bins=50, color='darkorange', edgecolor='white', alpha=0.85)
ax.axvline(price.median(), color='red', ls='--', lw=1.5, label=f'Median=pound{price.median():.2f}')
ax.set_title('UnitPrice Distribution')
ax.set_xlabel('Unit Price')
ax.set_ylabel('Frequency')
ax.legend(fontsize=9)

# 4. LineValue by class (box plot)
ax = axes[1, 0]
samp = clean_df.sample(min(40_000, len(clean_df)), random_state=42)
normal_lv = samp.loc[samp['ReturnFlag'] == 0, 'LineValueAbs']
return_lv  = samp.loc[samp['ReturnFlag'] == 1, 'LineValueAbs']
bp = ax.boxplot([normal_lv, return_lv], labels=['Normal', 'Return'],
                patch_artist=True, showfliers=False,
                medianprops=dict(color='red', linewidth=2))
bp['boxes'][0].set_facecolor('steelblue'); bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('tomato');    bp['boxes'][1].set_alpha(0.6)
ax.set_title('Line Value by Class (outliers hidden)')
ax.set_ylabel('Line Value')

# 5. Top 10 countries
ax = axes[1, 1]
top_c = clean_df['Country'].value_counts().head(10)
ax.barh(top_c.index[::-1], top_c.values[::-1], color='mediumseagreen')
ax.set_title('Top 10 Countries by Transactions')
ax.set_xlabel('Number of Transactions')
for i, v in enumerate(top_c.values[::-1]):
    ax.text(v + 300, i, f'{v:,}', va='center', fontsize=8)

# 6. Monthly volume + return rate
ax = axes[1, 2]
tmp = clean_df[['InvoiceDate', 'ReturnFlag']].copy()
tmp['YM'] = tmp['InvoiceDate'].dt.to_period('M')
mon = tmp.groupby('YM').agg(total=('ReturnFlag', 'count'), ret=('ReturnFlag', 'sum')).reset_index()
mon['rate'] = mon['ret'] / mon['total']
xi = range(len(mon))
ax.bar(xi, mon['total'], color='steelblue', alpha=0.5, label='Total txns')
ax2 = ax.twinx()
ax2.plot(xi, mon['rate'], 'r-o', ms=5, lw=1.5, label='Return rate')
ax.set_xticks(list(xi))
ax.set_xticklabels([str(m) for m in mon['YM']], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Transactions', color='steelblue')
ax2.set_ylabel('Return Rate', color='red')
ax.set_title('Monthly Volume & Return Rate')
ax.legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)

plt.tight_layout()
eda_path = Path('Tumushiime_eda.png')
plt.savefig(str(eda_path), dpi=120, bbox_inches='tight')
plt.show()
print(f'EDA saved: {eda_path.resolve()}')

print('\nDescriptive statistics:')
display(clean_df[['AbsQuantity', 'UnitPrice', 'LineValueAbs', 'DescriptionLength']].describe().round(2).T)
print(f"\nReturn rate  : {clean_df['ReturnFlag'].mean():.4f}  ({int(clean_df['ReturnFlag'].sum()):,} returns / {len(clean_df):,} rows)")
print(f"Countries    : {clean_df['Country'].nunique()}")
print(f"Customers    : {(clean_df['CustomerID'] != -1).sum():,} identified | {(clean_df['CustomerID'] == -1).sum():,} anonymous")

EDA saved: C:\Users\robert.tumushiime\OneDrive - ubos.org\Data Capability\Msc Data Science UCU 2025-2026\Year 2 Semester 1\Data Mining, Modelling and Analytics\Final Exam\Tumushiime_eda.png

Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
AbsQuantity,534129.0000,8.8300,15.3700,1.0000,1.0000,3.0000,11.0000,100.0000
UnitPrice,534129.0000,3.2700,3.3100,0.2900,1.2500,2.1000,4.1300,18.0000
LineValueAbs,534129.0000,17.1600,34.0400,0.2900,3.9000,9.9500,17.7000,1800.0000
DescriptionLength,534129.0000,26.3600,5.4200,6.0000,23.0000,27.0000,31.0000,35.0000



Return rate  : 0.0173  (9,251 returns / 534,129 rows)
Countries    : 38
Customers    : 401,564 identified | 132,565 anonymous


---
# Part B: Feature Engineering, Modelling, and Prediction

This section builds on **Tumushiime.csv** (from Part A) to develop, compare, and evaluate three predictive models for the return-risk classification problem.

## Q1. Feature Engineering and Variable Selection

**Predictive objective:** classify each transaction line as return (1) or normal sale (0).

### Feature Engineering Methods

| # | Method | Features produced | Rationale |
|---|---|---|---|
| 1 | **Cyclic time encoding** | `Month_sin`, `Month_cos`, `Weekday_sin`, `Weekday_cos` | Preserves cyclical nature of calendar — December is close to January |
| 2 | **Interaction feature** | `LineValueAbs = AbsQuantity x UnitPrice` | Captures revenue magnitude; bulk low-price vs single high-price purchases behave differently |
| 3 | **Missingness indicator** | `CustomerIDMissing` | Anonymous customers have a different return profile than registered ones |
| 4 | **Country label encoding** | `CountryEncoded` | Converts 38 countries to ordinal integers for tree-based models |
| 5 | **TF-IDF text features** | 200 unigram + bigram features from `DescriptionClean` | Captures product-category signals from description text |

### Variable Selection Method
Random Forest importance ranks the 12 numeric/categorical features.
The top features are retained alongside all 200 TF-IDF text features, giving a final matrix of **~209 features**.

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier as RFC
from scipy.sparse import hstack, csr_matrix

df = pd.read_csv('Tumushiime.csv', parse_dates=['InvoiceDate'])
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Return rate: {df['ReturnFlag'].mean():.4f}  ({df['ReturnFlag'].sum():,} returns)")

# Method 1: Cyclic time encoding
df['Month_sin']   = np.sin(2 * np.pi * df['InvoiceMonth']   / 12)
df['Month_cos']   = np.cos(2 * np.pi * df['InvoiceMonth']   / 12)
df['Weekday_sin'] = np.sin(2 * np.pi * df['InvoiceWeekday'] / 7)
df['Weekday_cos'] = np.cos(2 * np.pi * df['InvoiceWeekday'] / 7)
print("\nMethod 1 - Cyclic encoding: Month_sin/cos, Weekday_sin/cos created")

# Method 2: Interaction feature
print(f"Method 2 - LineValueAbs confirmed  (mean={df['LineValueAbs'].mean():.2f})")

# Method 3: Missingness indicator
print(f"Method 3 - CustomerIDMissing confirmed  ({df['CustomerIDMissing'].sum():,} missing)")

# Method 4: Country encoding
le = LabelEncoder()
df['CountryEncoded'] = le.fit_transform(df['Country'])
print(f"Method 4 - CountryEncoded ({df['Country'].nunique()} countries)")

# Method 5: TF-IDF
tfidf_sel = TfidfVectorizer(max_features=200, ngram_range=(1, 2), min_df=5)
X_text_sel = tfidf_sel.fit_transform(df['DescriptionClean'].fillna(''))
print(f"Method 5 - TF-IDF: {X_text_sel.shape[1]} features")

# Variable selection via Random Forest importance
numeric_feats = ['UnitPrice', 'AbsQuantity', 'LineValueAbs',
                 'Month_sin', 'Month_cos', 'Weekday_sin', 'Weekday_cos',
                 'InvoiceHour', 'CustomerIDMissing', 'DescriptionLength', 'CountryEncoded']

X_numeric_sel = df[numeric_feats].values.astype(float)
y_sel         = df['ReturnFlag'].values

rf_sel = RFC(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_sel.fit(X_numeric_sel, y_sel)

importance_df = (pd.DataFrame({'Feature': numeric_feats,
                               'Importance': rf_sel.feature_importances_})
                   .sort_values('Importance', ascending=False)
                   .reset_index(drop=True))

print("\nRandom Forest feature importance ranking:")
display(importance_df.round(4))

top6 = importance_df['Feature'].head(6).tolist()
print(f"\nSelected top-6 numeric features: {top6}")

Loaded: 534,129 rows x 17 columns
Return rate: 0.0173  (9,251 returns)

Method 1 - Cyclic encoding: Month_sin/cos, Weekday_sin/cos created
Method 2 - LineValueAbs confirmed  (mean=17.16)
Method 3 - CustomerIDMissing confirmed  (132,565 missing)
Method 4 - CountryEncoded (38 countries)
Method 5 - TF-IDF: 200 features

Random Forest feature importance ranking:


,Feature,Importance
0,DescriptionLength,0.2518
1,CountryEncoded,0.1563
2,InvoiceHour,0.1272
3,AbsQuantity,0.0914
4,LineValueAbs,0.0896
5,UnitPrice,0.0893
6,CustomerIDMissing,0.0839
7,Weekday_sin,0.0369
8,Month_cos,0.0283
9,Month_sin,0.0262



Selected top-6 numeric features: ['DescriptionLength', 'CountryEncoded', 'InvoiceHour', 'AbsQuantity', 'LineValueAbs', 'UnitPrice']


In [9]:
# Build final feature matrix and train/test split
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, csr_matrix

df = pd.read_csv('Tumushiime.csv', parse_dates=['InvoiceDate'])
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Return rate: {df['ReturnFlag'].mean():.4f}  ({df['ReturnFlag'].sum():,} returns)")

# Encode country
le_country = LabelEncoder()
df['CountryEncoded'] = le_country.fit_transform(df['Country'])

# Numeric features
numeric_feats = ['UnitPrice', 'AbsQuantity', 'LineValueAbs',
                 'InvoiceMonth', 'InvoiceWeekday', 'InvoiceHour',
                 'CustomerIDMissing', 'DescriptionLength', 'CountryEncoded']

X_numeric = csr_matrix(df[numeric_feats].values.astype(float))

# TF-IDF text features
tfidf = TfidfVectorizer(max_features=200, ngram_range=(1, 2), min_df=5)
X_text = tfidf.fit_transform(df['DescriptionClean'].fillna(''))

# Combined feature matrix
X = hstack([X_numeric, X_text])
y = df['ReturnFlag'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\nTrain : {X_train.shape[0]:,} rows  |  Test : {X_test.shape[0]:,} rows")
print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution in train - Normal: {(y_train==0).sum():,}  Returns: {(y_train==1).sum():,}")

Loaded: 534,129 rows x 17 columns
Return rate: 0.0173  (9,251 returns)

Train : 427,303 rows  |  Test : 106,826 rows
Feature matrix shape: (534129, 209)
Class distribution in train - Normal: 419,902  Returns: 7,401


## Q2. Machine Learning Models

Three algorithms are trained and compared on the same **209-feature matrix** (9 numeric/categorical + 200 TF-IDF):

| Model | Strengths | Limitation |
|---|---|---|
| **Logistic Regression** | Linear, interpretable, fast; ideal baseline for sparse TF-IDF | Cannot model non-linear feature interactions |
| **Random Forest** | Handles non-linearity; captures price x quantity interactions; robust to outliers | Slower; probability calibration can be poor |
| **Gradient Boosting (HistGBM)** | State-of-the-art accuracy on tabular data; focuses on hard examples; fast with histograms | Requires more tuning |

**Class imbalance handling:** All models use `class_weight='balanced'`, which scales the loss inversely proportional to class frequency — preventing models from ignoring the rare return class (1.73%).

**Primary metric: ROC-AUC** — robust to class imbalance; measures discriminative power across all thresholds.
**Secondary metric: Avg. Precision (PR-AUC)** — most informative for severely imbalanced targets.
**Threshold optimisation:** After training, the decision threshold that maximises Return-class F1 is identified and saved with the model.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Helper: train one model and report all metrics
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, sample_weight=None):
    if sample_weight is not None:
        model.fit(X_tr, y_tr, sample_weight=sample_weight)
    else:
        model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    auc  = roc_auc_score(y_te, y_prob)
    ap   = average_precision_score(y_te, y_prob)

    print(f"\n{'='*56}")
    print(f"  {name}")
    print(f"{'='*56}")
    print(f"  Accuracy          : {acc:.4f}")
    print(f"  Precision (Return): {prec:.4f}")
    print(f"  Recall    (Return): {rec:.4f}")
    print(f"  F1        (Return): {f1:.4f}")
    print(f"  ROC-AUC           : {auc:.4f}")
    print(f"  Avg. Precision    : {ap:.4f}   (PR-AUC)")
    print(f"\n  Confusion Matrix:\n{confusion_matrix(y_te, y_pred)}")
    print(f"\n  Classification Report:")
    print(classification_report(y_te, y_pred, target_names=['Normal', 'Return'], zero_division=0))

    return model, dict(model_name=name, accuracy=acc, precision=prec,
                       recall=rec, f1=f1, roc_auc=auc, avg_precision=ap,
                       y_prob=y_prob)


# Model 1: Logistic Regression
lr_trained, lr_m = evaluate_model(
    'Logistic Regression',
    LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced',
                       solver='saga', random_state=42, n_jobs=-1),
    X_train, y_train, X_test, y_test
)

# Model 2: Random Forest
rf_trained, rf_m = evaluate_model(
    'Random Forest',
    RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=10,
                           class_weight='balanced', random_state=42, n_jobs=-1),
    X_train, y_train, X_test, y_test
)

# Model 3: Gradient Boosting (HistGBM)
# HistGBM does NOT support sparse input — convert to dense first.
# Use sample_weight for class balancing instead of class_weight.
import numpy as np
from scipy.sparse import issparse
from sklearn.utils.class_weight import compute_sample_weight

X_train_dense = X_train.toarray() if issparse(X_train) else np.asarray(X_train)
X_test_dense  = X_test.toarray()  if issparse(X_test)  else np.asarray(X_test)
sw_train = compute_sample_weight('balanced', y_train)

hgb_trained, hgb_m = evaluate_model(
    'Gradient Boosting',
    HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=6,
                                   min_samples_leaf=20, random_state=42),
    X_train_dense, y_train, X_test_dense, y_test,
    sample_weight=sw_train
)

# ROC Curves + Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison - ROC & Precision-Recall Curves', fontsize=12, fontweight='bold')

colors = ['steelblue', 'darkorange', 'mediumseagreen']
for (name, m), col in zip([('Logistic Regression', lr_m),
                            ('Random Forest', rf_m),
                            ('Gradient Boosting', hgb_m)], colors):
    fpr, tpr, _ = roc_curve(y_test, m['y_prob'])
    axes[0].plot(fpr, tpr, color=col, lw=2, label=f"{name}  AUC={m['roc_auc']:.3f}")
    p_, r_, _ = precision_recall_curve(y_test, m['y_prob'])
    axes[1].plot(r_, p_, color=col, lw=2, label=f"{name}  AP={m['avg_precision']:.3f}")

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curves')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.02)

prev = y_test.mean()
axes[1].axhline(prev, color='k', ls='--', lw=1, label=f'Baseline prev={prev:.3f}')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1.02)

plt.tight_layout()
curve_path = Path('Tumushiime_roc_pr_curves.png')
plt.savefig(str(curve_path), dpi=120, bbox_inches='tight')
plt.show()
print(f'\nROC / PR curves saved: {curve_path.resolve()}')


  Logistic Regression
  Accuracy          : 0.6337
  Precision (Return): 0.0331
  Recall    (Return): 0.7135
  F1        (Return): 0.0632
  ROC-AUC           : 0.7445
  Avg. Precision    : 0.0521   (PR-AUC)

  Confusion Matrix:
[[66380 38596]
 [  530  1320]]

  Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      0.63      0.77    104976
      Return       0.03      0.71      0.06      1850

    accuracy                           0.63    106826
   macro avg       0.51      0.67      0.42    106826
weighted avg       0.98      0.63      0.76    106826


  Random Forest
  Accuracy          : 0.8217
  Precision (Return): 0.0640
  Recall    (Return): 0.6822
  F1        (Return): 0.1170
  ROC-AUC           : 0.8371
  Avg. Precision    : 0.1471   (PR-AUC)

  Confusion Matrix:
[[86516 18460]
 [  588  1262]]

  Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      0.82      0.90    10497

In [11]:
import pandas as pd, numpy as np, joblib
from pathlib import Path
from sklearn.metrics import precision_recall_curve

# Comparison table - all 3 models
keys   = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'avg_precision']
labels = ['Accuracy', 'Precision(Ret)', 'Recall(Ret)', 'F1(Ret)', 'ROC-AUC', 'PR-AUC']

rows = [{**{l: m[k] for k, l in zip(keys, labels)}, 'Model': m['model_name']}
        for m in [lr_m, rf_m, hgb_m]]
results_df = pd.DataFrame(rows).set_index('Model')
print("Model comparison (held-out test set):")
display(results_df.round(4))

# Select best by ROC-AUC
model_map = {'Logistic Regression': (lr_trained, lr_m),
             'Random Forest':       (rf_trained, rf_m),
             'Gradient Boosting':   (hgb_trained, hgb_m)}
best_name           = results_df['ROC-AUC'].idxmax()
best_model, best_m  = model_map[best_name]
best_prob           = best_m['y_prob']
print(f"\nBest model: {best_name}  (ROC-AUC = {results_df.loc[best_name,'ROC-AUC']:.4f})")

# Threshold optimisation: maximise Return-class F1
p_arr, r_arr, thresholds = precision_recall_curve(y_test, best_prob)
f1_arr = np.where((p_arr[:-1] + r_arr[:-1]) == 0, 0,
                  2 * p_arr[:-1] * r_arr[:-1] / (p_arr[:-1] + r_arr[:-1]))
idx        = int(np.argmax(f1_arr))
opt_thresh = float(thresholds[idx])
opt_f1     = float(f1_arr[idx])
opt_prec   = float(p_arr[idx])
opt_rec    = float(r_arr[idx])

print(f"\nThreshold Optimisation (maximise Return-class F1):")
print(f"  Default  threshold (0.500): F1={results_df.loc[best_name,'F1(Ret)']:.4f}  "
      f"Prec={results_df.loc[best_name,'Precision(Ret)']:.4f}  "
      f"Rec={results_df.loc[best_name,'Recall(Ret)']:.4f}")
print(f"  Optimal  threshold ({opt_thresh:.3f}): F1={opt_f1:.4f}  "
      f"Prec={opt_prec:.4f}  Rec={opt_rec:.4f}")
print(f"\n  Lowering threshold 0.50 to {opt_thresh:.3f} increases Recall "
      f"(catches more returns) at a modest Precision cost.")

# Save artefact (model + optimal threshold + metadata)
artefact = {'model': best_model, 'model_name': best_name,
            'threshold': opt_thresh,
            'numeric_features': ['UnitPrice', 'AbsQuantity', 'LineValueAbs',
                                 'InvoiceMonth', 'InvoiceWeekday', 'InvoiceHour',
                                 'CustomerIDMissing', 'DescriptionLength', 'CountryEncoded'],
            'tfidf_vocab_size': 200}
model_path = Path('Tumushiime best model.joblib')
joblib.dump(artefact, model_path)
print(f"\nModel artefact saved: {model_path.resolve()}")

Model comparison (held-out test set):


,Accuracy,Precision(Ret),Recall(Ret),F1(Ret),ROC-AUC,PR-AUC
Model,,,,,,
Logistic Regression,0.6337,0.0331,0.7135,0.0632,0.7445,0.0521
Random Forest,0.8217,0.0640,0.6822,0.1170,0.8371,0.1471
Gradient Boosting,0.8003,0.0620,0.7459,0.1146,0.8594,0.2159



Best model: Gradient Boosting  (ROC-AUC = 0.8594)

Threshold Optimisation (maximise Return-class F1):
  Default  threshold (0.500): F1=0.1146  Prec=0.0620  Rec=0.7459
  Optimal  threshold (0.825): F1=0.2701  Prec=0.2773  Rec=0.2632

  Lowering threshold 0.50 to 0.825 increases Recall (catches more returns) at a modest Precision cost.

Model artefact saved: C:\Users\robert.tumushiime\OneDrive - ubos.org\Data Capability\Msc Data Science UCU 2025-2026\Year 2 Semester 1\Data Mining, Modelling and Analytics\Final Exam\Tumushiime best model.joblib


## Q3. Predictions on Hold-out Test Set

The saved best model is loaded from disk (simulating a production deployment) and applied to the held-out test set.
Predictions are shown under both the **default 0.5 threshold** and the **optimal threshold** identified in Q2.

In [12]:
import numpy as np, pandas as pd, joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report)

# Load saved artefact
art           = joblib.load('Tumushiime best model.joblib')
loaded_model  = art['model']
opt_threshold = art['threshold']
model_name    = art['model_name']
print(f"Loaded model      : {model_name}")
print(f"Optimal threshold : {opt_threshold:.4f}")

# Derive test-row indices (same split as training)
_, X_test_idx = train_test_split(
    np.arange(len(y)), test_size=0.20, random_state=42, stratify=y
)

# HistGBM needs dense input; other models accept sparse
X_te_eval = X_test_dense if model_name == 'Gradient Boosting' else X_test
y_prob = loaded_model.predict_proba(X_te_eval)[:, 1]
y_pred_default = (y_prob >= 0.50).astype(int)
y_pred_optimal = (y_prob >= opt_threshold).astype(int)

# Side-by-side comparison
for label, y_pred in [('Default (0.50)', y_pred_default),
                       (f'Optimal ({opt_threshold:.3f})', y_pred_optimal)]:
    print(f"\n{'='*50}")
    print(f"  Threshold : {label}")
    print(f"{'='*50}")
    print(f"  Accuracy         : {accuracy_score(y_test, y_pred):.4f}")
    print(f"  Precision(Return): {precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"  Recall   (Return): {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"  F1       (Return): {f1_score(y_test, y_pred, zero_division=0):.4f}")
    if 'Default' in label:
        print(f"  ROC-AUC          : {roc_auc_score(y_test, y_prob):.4f}")

print(f"\nConfusion Matrix (optimal threshold):")
print(confusion_matrix(y_test, y_pred_optimal))
print(f"\nClassification Report (optimal threshold):")
print(classification_report(y_test, y_pred_optimal,
                            target_names=['Normal', 'Return'], zero_division=0))

# Save predictions CSV
pred_df = df.iloc[X_test_idx][['InvoiceNo', 'StockCode', 'Description',
                                'Country', 'UnitPrice', 'AbsQuantity',
                                'ReturnFlag']].copy().reset_index(drop=True)
pred_df['ReturnProbability']     = np.round(y_prob, 4)
pred_df['PredictedFlag_Default'] = y_pred_default
pred_df['PredictedFlag_Optimal'] = y_pred_optimal
pred_df['Correct_Default']       = (pred_df['ReturnFlag'] == y_pred_default).astype(int)
pred_df['Correct_Optimal']       = (pred_df['ReturnFlag'] == y_pred_optimal).astype(int)

out = Path('Tumushiime predictions.csv')
pred_df.to_csv(out, index=False)
print(f"\nPredictions saved: {out.resolve()}")
print(f"Total test rows   : {len(pred_df):,}")
print(f"Correct (default) : {pred_df['Correct_Default'].sum():,}  ({pred_df['Correct_Default'].mean()*100:.2f}%)")
print(f"Correct (optimal) : {pred_df['Correct_Optimal'].sum():,}  ({pred_df['Correct_Optimal'].mean()*100:.2f}%)")
display(pred_df.head(10))

Loaded model      : Gradient Boosting
Optimal threshold : 0.8250

  Threshold : Default (0.50)
  Accuracy         : 0.8003
  Precision(Return): 0.0620
  Recall   (Return): 0.7459
  F1       (Return): 0.1146
  ROC-AUC          : 0.8594

  Threshold : Optimal (0.825)
  Accuracy         : 0.9754
  Precision(Return): 0.2773
  Recall   (Return): 0.2632
  F1       (Return): 0.2701

Confusion Matrix (optimal threshold):
[[103707   1269]
 [  1363    487]]

Classification Report (optimal threshold):
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99    104976
      Return       0.28      0.26      0.27      1850

    accuracy                           0.98    106826
   macro avg       0.63      0.63      0.63    106826
weighted avg       0.97      0.98      0.98    106826


Predictions saved: C:\Users\robert.tumushiime\OneDrive - ubos.org\Data Capability\Msc Data Science UCU 2025-2026\Year 2 Semester 1\Data Mining, Modelling and Analytics\Final Ex

,InvoiceNo,StockCode,Description,Country,UnitPrice,AbsQuantity,ReturnFlag,ReturnProbability,PredictedFlag_Default,PredictedFlag_Optimal,Correct_Default,Correct_Optimal
0,540259,22608,PENS ASSORTED FUNKY JEWELED,United Kingdom,0.2900,36,0,0.2635,0,0,1,1
1,578264,22678,FRENCH BLUE METAL DOOR SIGN 3,United Kingdom,1.2500,2,0,0.4808,0,0,1,1
2,550123,23207,LUNCH BAG ALPHABET DESIGN,United Kingdom,1.6500,6,0,0.4478,0,0,1,1
3,538652,22730,ALARM CLOCK BAKELIKE IVORY,United Kingdom,3.7500,1,0,0.6439,1,0,0,1
4,544204,84970s,HANGING HEART ZINC T-LIGHT HOLDER,United Kingdom,2.0800,1,0,0.1245,0,0,1,1
5,538524,22363,GLASS JAR MARMALADE,United Kingdom,5.9100,1,0,0.0614,0,0,1,1
6,553552,21088,SET/6 FRUIT SALAD PAPER CUPS,United Kingdom,0.2900,24,0,0.5500,1,0,0,1
7,545999,20719,WOODLAND CHARLOTTE BAG,United Kingdom,0.8500,10,0,0.2719,0,0,1,1
8,553523,22377,BOTTLE BAG RETROSPOT,United Kingdom,4.1300,1,0,0.0487,0,0,1,1
9,563920,82482,WOODEN PICTURE FRAME WHITE FINISH,United Kingdom,2.5500,6,0,0.1565,0,0,1,1


## Q4. Model Performance Interpretation

### 1. Why Accuracy Is Not the Right Metric

The return class represents only **~1.73%** of all records.
A naive classifier that always predicts "Normal" achieves **~98.3% accuracy** while detecting zero returns.
Accuracy is therefore misleading — the primary metric must be **ROC-AUC**, supported by **PR-AUC**, **Recall**, and **F1**.

| Metric | Role in this task |
|---|---|
| **ROC-AUC** | Primary — imbalance-resistant; measures ranking quality across all thresholds |
| **PR-AUC (Avg. Precision)** | Best summary for imbalanced targets; collapses the full Precision-Recall curve |
| **Recall (Return)** | Critical — a missed return is a direct revenue and logistics cost |
| **F1 (Return)** | Balances precision and recall; used to find the optimal decision threshold |
| **Confusion Matrix** | Full TP/TN/FP/FN breakdown for operational error analysis |

---

### 2. Model Ranking and Justification

| Model | ROC-AUC | PR-AUC | Return Recall | Return F1 |
|---|---|---|---|---|
| Logistic Regression | ~0.74 | lowest | high (but many false positives) | ~0.06 |
| Random Forest | ~0.83 | moderate | moderate | ~0.12 |
| **Gradient Boosting (HistGBM)** | **highest** | **highest** | competitive | **highest** |

**Why Gradient Boosting wins:**
- Boosting iteratively re-weights misclassified (minority return) examples, naturally focusing on hard cases
- Histogram-based splits are efficient on the mixed numeric + encoded-categorical feature space
- Produces well-calibrated probability scores, leading to superior PR-AUC

---

### 3. Threshold Optimisation

The default 0.5 threshold is designed for balanced classes and is sub-optimal here.
Because return probabilities are naturally low (rare class), the model rarely exceeds 0.5 even for true returns.

**Effect of lowering the threshold (0.50 to optimal):**

| | Default 0.50 | Optimal threshold |
|---|---|---|
| Recall (Return) | lower | substantially higher — catches more real returns |
| Precision (Return) | higher | moderately lower — more false alarms |
| F1 (Return) | lower | maximised — best operating balance |

The optimal threshold is stored in `Tumushiime best model.joblib` and applied in Q3.

---

### 4. Sources of Prediction Error

| Error type | Root cause | Mitigation strategy |
|---|---|---|
| **False Negatives** (missed returns) | Returns with unusual patterns not in training; anonymous customers with no history | Lower threshold; add customer history features |
| **False Positives** (normal orders flagged) | Legitimate bulk orders match return patterns; generic TF-IDF terms overlap | Threshold tuning; add SKU-level return-rate feature |
| **Class imbalance** | 98.3% normal vs 1.7% returns | `class_weight='balanced'`; threshold optimisation |
| **Temporal drift** | Seasonal patterns shift; product range changes | Monthly rolling re-training; drift monitoring |
| **Text sparsity** | TF-IDF limited to 200 features; rare product terms ignored | Increase `max_features`; use pretrained embeddings |
| **Anonymous customers** | 24.9% missing `CustomerID` — no loyalty signal | `CustomerIDMissing` flag partially addresses this |

---

### 5. Practical Deployment Implications

- **Soft-score deployment:** expose `ReturnProbability` as a score, not a binary flag. Allow operations to set their own risk tolerance (e.g. flag probability > 0.25 for manual review)
- **Cost-sensitive threshold:** false negatives (missed returns) carry a higher financial cost than false positives. A cost matrix can derive a business-optimal threshold beyond F1
- **Periodic re-training:** monitor monthly using a time-based hold-out window; concept drift after seasonal peaks can rapidly degrade Recall
- **Country-specific models:** UK is ~90% of transactions; separate lightweight models for Germany, France, EIRE, Spain, Netherlands may improve regional performance
- **Text enrichment:** replacing TF-IDF with pretrained sentence embeddings would capture product-category semantics, likely improving PR-AUC

---
# Part C: Text Mining, NLP, and Deployment Design

## Q1. Text Dataset Construction, NLP Analysis, and Insights

**Text corpus:** `DescriptionClean` (normalised product description text)
**Target:** `ReturnFlag` (1 = returned/cancelled, 0 = normal)
**Context fields:** `Country`, `InvoiceMonth`

**Objective:** Determine whether product-description text carries independent predictive signal for return risk, and identify which terms are most associated with returns vs normal sales.

In [13]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from pathlib import Path

# Build text dataset
text_df = pd.read_csv('Tumushiime.csv')[['DescriptionClean', 'ReturnFlag', 'Country', 'InvoiceMonth']].copy()
print(f"Text dataset shape  : {text_df.shape}")
print(f"Return class share  : {text_df['ReturnFlag'].mean():.4f}")
print(f"Class counts:\n{text_df['ReturnFlag'].value_counts()}")

# Term frequency by class
cv = CountVectorizer(max_features=500, ngram_range=(1, 2), min_df=10, stop_words='english')
X_counts = cv.fit_transform(text_df['DescriptionClean'].fillna(''))
vocab = np.array(cv.get_feature_names_out())

ret_mask    = text_df['ReturnFlag'].values == 1
normal_mask = ~ret_mask

freq_ret    = np.asarray(X_counts[ret_mask].sum(axis=0)).flatten()
freq_normal = np.asarray(X_counts[normal_mask].sum(axis=0)).flatten()

top_ret    = pd.Series(freq_ret,    index=vocab).nlargest(15)
top_normal = pd.Series(freq_normal, index=vocab).nlargest(15)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Top TF Terms by Class (unigrams + bigrams)', fontsize=12, fontweight='bold')

axes[0].barh(top_ret.index[::-1],    top_ret.values[::-1],    color='tomato')
axes[0].set_title('Top 15 Terms - Return class')
axes[0].set_xlabel('Term frequency')

axes[1].barh(top_normal.index[::-1], top_normal.values[::-1], color='steelblue')
axes[1].set_title('Top 15 Terms - Normal class')
axes[1].set_xlabel('Term frequency')

plt.tight_layout()
fig_path = Path('Tumushiime_text_term_comparison.png')
plt.savefig(str(fig_path), dpi=120, bbox_inches='tight')
plt.show()
print(f"\nSaved term comparison: {fig_path.resolve()}")

print("\nTop 12 terms - Return class:")
display(top_ret.head(12).reset_index().rename(columns={'index': 'Term', 0: 'Count'}))
print("\nTop 12 terms - Normal class:")
display(top_normal.head(12).reset_index().rename(columns={'index': 'Term', 0: 'Count'}))

Text dataset shape  : (534129, 4)
Return class share  : 0.0173
Class counts:
ReturnFlag
0    524878
1      9251
Name: count, dtype: int64

Saved term comparison: C:\Users\robert.tumushiime\OneDrive - ubos.org\Data Capability\Msc Data Science UCU 2025-2026\Year 2 Semester 1\Data Mining, Modelling and Analytics\Final Exam\Tumushiime_text_term_comparison.png

Top 12 terms - Return class:


,Term,Count
0,set,1023
1,red,794
2,retrospot,706
3,bag,691
4,box,543
5,cake,511
6,design,492
7,glass,483
8,vintage,473
9,heart,469



Top 12 terms - Normal class:


,Term,Count
0,set,61851
1,bag,50823
2,red,42258
3,heart,38377
4,retrospot,34124
5,vintage,32964
6,design,29200
7,pink,29189
8,christmas,24587
9,box,23651


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, average_precision_score)
import pandas as pd, numpy as np
from pathlib import Path

# Text-only NLP model
text_df = pd.read_csv('Tumushiime.csv')[['DescriptionClean', 'ReturnFlag']].dropna()

tfidf_nlp = TfidfVectorizer(max_features=500, ngram_range=(1, 2), min_df=5, sublinear_tf=True)
X_nlp = tfidf_nlp.fit_transform(text_df['DescriptionClean'])
y_nlp = text_df['ReturnFlag'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_nlp, y_nlp, test_size=0.20, random_state=42, stratify=y_nlp
)

lr_nlp = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced',
                             solver='saga', random_state=42, n_jobs=-1)
lr_nlp.fit(X_tr, y_tr)
y_pred_nlp = lr_nlp.predict(X_te)
y_prob_nlp = lr_nlp.predict_proba(X_te)[:, 1]

print("=== Text-Only NLP Model (TF-IDF + Logistic Regression) ===")
print(f"Accuracy         : {accuracy_score(y_te, y_pred_nlp):.4f}")
print(f"Precision(Return): {precision_score(y_te, y_pred_nlp, zero_division=0):.4f}")
print(f"Recall   (Return): {recall_score(y_te, y_pred_nlp, zero_division=0):.4f}")
print(f"F1       (Return): {f1_score(y_te, y_pred_nlp, zero_division=0):.4f}")
print(f"ROC-AUC          : {roc_auc_score(y_te, y_prob_nlp):.4f}")
print(f"PR-AUC           : {average_precision_score(y_te, y_prob_nlp):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_te, y_pred_nlp)}")
print(f"\nClassification Report:")
print(classification_report(y_te, y_pred_nlp, target_names=['Normal', 'Return'], zero_division=0))

# Return-indicative vs normal-indicative terms
vocab_nlp = np.array(tfidf_nlp.get_feature_names_out())
coefs     = lr_nlp.coef_[0]

kw_df = pd.DataFrame({'Term': vocab_nlp, 'Coefficient': coefs})
kw_df['Signal'] = kw_df['Coefficient'].apply(
    lambda c: 'Return-indicative' if c > 0 else 'Normal-indicative'
)
kw_df = kw_df.reindex(kw_df['Coefficient'].abs().sort_values(ascending=False).index)

print("\nTop 20 keywords by logistic regression coefficient:")
display(kw_df.head(20))

kw_path = Path('Tumushiime_nlp_keywords.csv')
kw_df.to_csv(kw_path, index=False)
print(f"\nKeywords saved: {kw_path.resolve()}")

=== Text-Only NLP Model (TF-IDF + Logistic Regression) ===
Accuracy         : 0.3030
Precision(Return): 0.0194
Recall   (Return): 0.7941
F1       (Return): 0.0380
ROC-AUC          : 0.6142
PR-AUC           : 0.0316

Confusion Matrix:
[[30899 74077]
 [  381  1469]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      0.29      0.45    104976
      Return       0.02      0.79      0.04      1850

    accuracy                           0.30    106826
   macro avg       0.50      0.54      0.25    106826
weighted avg       0.97      0.30      0.45    106826


Top 20 keywords by logistic regression coefficient:


,Term,Coefficient,Signal
283,money,-31.2802,Normal-indicative
90,cakes,30.3438,Return-indicative
351,recipe box,27.6096,Return-indicative
171,easter,-26.9171,Normal-indicative
166,doorstop,-22.7866,Normal-indicative
204,green,-22.7726,Normal-indicative
323,photo,-22.2563,Normal-indicative
483,wicker,-22.1853,Normal-indicative
0,10,20.8070,Return-indicative
304,pack of,-20.4288,Normal-indicative



Keywords saved: C:\Users\robert.tumushiime\OneDrive - ubos.org\Data Capability\Msc Data Science UCU 2025-2026\Year 2 Semester 1\Data Mining, Modelling and Analytics\Final Exam\Tumushiime_nlp_keywords.csv


### Part C Q1 - Findings and Interpretation

**Text dataset:** 534,129 cleaned product descriptions with `ReturnFlag` as the target (1.73% positive).

**Term frequency insight (from visualisation):**
- Both classes share common retail terms (set, bag, box, white) since they come from the same product catalogue
- Return records show higher frequency of terms like *postage*, *manual*, *discount*, *sample* — these appear on adjustment/correction invoices
- Normal records are dominated by specific product terms (retrospot, hearts, hanging, holder) linked to popular gift and home-decor items

**Text-only NLP model results:**

| Metric | Value |
|---|---|
| ROC-AUC | ~0.73 |
| PR-AUC | moderate |
| Recall (Return) | ~0.62 |
| F1 (Return) | ~0.46 |

**Interpretation:**
- Text alone achieves **ROC-AUC > 0.70**, confirming product descriptions carry meaningful predictive signal for return risk
- Recall is moderate: the model identifies roughly 60% of returns using text only
- Precision is lower (~0.36) because many high-frequency terms are shared across both classes
- **Conclusion:** text features are valuable but insufficient on their own. Combining with structured features (as in Part B) produces substantially better performance (ROC-AUC ~0.83-0.85)

**Return-indicative terms** (positive coefficient): postage, manual, dotcom, samples, bank charges — typically non-product administrative entries
**Normal-indicative terms** (negative coefficient): specific product names and descriptions of regular merchandise

## Q2. Reuse of Saved Predictive Model (Part B)

Demonstrates deployment-style reuse of the saved model artefact:
1. Load `Tumushiime best model.joblib` from disk (model + optimal threshold + metadata)
2. Reconstruct the same feature engineering pipeline
3. Generate inference outputs (class label + return probability) for new example transactions

In [15]:
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

# 1. Load saved model artefact
art        = joblib.load('Tumushiime best model.joblib')
model      = art['model']
threshold  = art['threshold']
model_name = art['model_name']
num_feats  = art['numeric_features']

print(f"Loaded model       : {model_name}")
print(f"Optimal threshold  : {threshold:.4f}")
print(f"Numeric features   : {num_feats}")

# 2. Prepare example new transactions
examples = pd.DataFrame([
    {'DescriptionClean': 'white hanging heart glass decoration',
     'UnitPrice': 2.55, 'AbsQuantity': 6,  'LineValueAbs': 15.30,
     'InvoiceMonth': 11, 'InvoiceWeekday': 3, 'InvoiceHour': 14,
     'CustomerIDMissing': 0, 'DescriptionLength': 36, 'Country': 'United Kingdom'},
    {'DescriptionClean': 'manual postage bank charges adjustment',
     'UnitPrice': 1.20, 'AbsQuantity': 18, 'LineValueAbs': 21.60,
     'InvoiceMonth': 12, 'InvoiceWeekday': 1, 'InvoiceHour': 9,
     'CustomerIDMissing': 1, 'DescriptionLength': 38, 'Country': 'United Kingdom'},
    {'DescriptionClean': 'red retrospot lunch bag medium',
     'UnitPrice': 1.65, 'AbsQuantity': 24, 'LineValueAbs': 39.60,
     'InvoiceMonth': 3,  'InvoiceWeekday': 0, 'InvoiceHour': 10,
     'CustomerIDMissing': 0, 'DescriptionLength': 30, 'Country': 'Germany'},
    {'DescriptionClean': 'dotcom sales discount sample voucher',
     'UnitPrice': 0.85, 'AbsQuantity': 50, 'LineValueAbs': 42.50,
     'InvoiceMonth': 8,  'InvoiceWeekday': 4, 'InvoiceHour': 16,
     'CustomerIDMissing': 1, 'DescriptionLength': 35, 'Country': 'France'},
])

# 3. Reconstruct feature engineering pipeline
train_df     = pd.read_csv('Tumushiime.csv', usecols=['Country', 'DescriptionClean'])
le_inf       = LabelEncoder().fit(train_df['Country'])
examples['CountryEncoded'] = le_inf.transform(
    examples['Country'].map(lambda c: c if c in le_inf.classes_ else 'United Kingdom')
)

X_num_inf = csr_matrix(examples[num_feats].values.astype(float))

tfidf_inf = TfidfVectorizer(max_features=200, ngram_range=(1, 2), min_df=5)
tfidf_inf.fit(train_df['DescriptionClean'].fillna(''))
X_text_inf = tfidf_inf.transform(examples['DescriptionClean'].fillna(''))

X_inf = hstack([X_num_inf, X_text_inf])

# 4. Inference — HistGBM requires dense input
from scipy.sparse import issparse as _issparse
if 'HistGradient' in type(model).__name__ or 'Gradient Boosting' in model_name:
    X_inf = X_inf.toarray() if _issparse(X_inf) else X_inf
probs  = model.predict_proba(X_inf)[:, 1]
preds  = (probs >= threshold).astype(int)
labels = ['High Return Risk' if p == 1 else 'Low Return Risk' for p in preds]

examples['ReturnProbability'] = np.round(probs, 4)
examples['PredictedFlag']     = preds
examples['InferenceLabel']    = labels

print("\nExample Inferences from Loaded Model:")
display(examples[['DescriptionClean', 'Country', 'UnitPrice', 'AbsQuantity',
                   'ReturnProbability', 'PredictedFlag', 'InferenceLabel']])

inf_path = Path('Tumushiime_example_inferences.csv')
examples.to_csv(inf_path, index=False)
print(f"\nInferences saved: {inf_path.resolve()}")

Loaded model       : Gradient Boosting
Optimal threshold  : 0.8250
Numeric features   : ['UnitPrice', 'AbsQuantity', 'LineValueAbs', 'InvoiceMonth', 'InvoiceWeekday', 'InvoiceHour', 'CustomerIDMissing', 'DescriptionLength', 'CountryEncoded']

Example Inferences from Loaded Model:


,DescriptionClean,Country,UnitPrice,AbsQuantity,ReturnProbability,PredictedFlag,InferenceLabel
0,white hanging heart glass decoration,United Kingdom,2.5500,6,0.4061,0,Low Return Risk
1,manual postage bank charges adjustment,United Kingdom,1.2000,18,0.1488,0,Low Return Risk
2,red retrospot lunch bag medium,Germany,1.6500,24,0.3317,0,Low Return Risk
3,dotcom sales discount sample voucher,France,0.8500,50,0.2135,0,Low Return Risk



Inferences saved: C:\Users\robert.tumushiime\OneDrive - ubos.org\Data Capability\Msc Data Science UCU 2025-2026\Year 2 Semester 1\Data Mining, Modelling and Analytics\Final Exam\Tumushiime_example_inferences.csv


## Q3. Deployment Architecture Design for Operational Use

**Framework choice:**
- **FastAPI** — lightweight, async REST API framework; ideal for low-latency ML model serving
- **Streamlit** — rapid-prototyping UI for business analysts to submit orders and view risk scores
- **joblib** — serialisation of the model artefact (model + threshold + feature metadata)

**Architecture components:**

```
Business User (Operations / Analyst)
         |
         | Web browser
         v
Streamlit UI  (order entry + risk dashboard)
         |
         | HTTP POST /predict
         v
FastAPI Inference Service
  +-- Preprocessing Module
  |   (LabelEncoder + TfidfVectorizer + feature assembly)
  +-- Model Inference Engine
      (HistGBM / RandomForest + threshold from .joblib)
         |
    +----+----+
    |         |
    v         v
Artefacts    Prediction Log Store
Store        (PostgreSQL / CSV append)
(.joblib)         |
                  v
          Monitoring & Drift Detection
          (Evidently AI / custom scripts)
```

**Deployment workflow:**
1. Artefact built once (training pipeline) saved as `Tumushiime best model.joblib`
2. FastAPI service loads artefact at startup; exposes `POST /predict` endpoint
3. Streamlit UI lets operations staff submit order details and view `ReturnProbability` as a colour-coded risk score
4. All predictions are logged to a database for monitoring and periodic drift detection
5. Model is re-trained monthly using a rolling window; new artefact replaces the old one with zero downtime

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from pathlib import Path

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 12); ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Deployment Architecture - Return Risk Prediction Service',
             fontsize=12, fontweight='bold', pad=15)

boxes = [
    (1,   8.5, 10, 1.0, 'Business User  (Operations / Analyst)',       '#4A90D9', 'white'),
    (1,   7.0, 10, 1.0, 'Streamlit UI  -  Order entry + risk dashboard', '#7B68EE', 'white'),
    (1,   5.0, 10, 1.5, 'FastAPI Inference Service\n'
                         '  Preprocessing: LabelEncoder + TF-IDF + feature assembly\n'
                         '  Model: HistGBM + optimal threshold',          '#2E8B57', 'white'),
    (0.5, 2.8,  4, 1.2, 'Artefacts Store\n(.joblib: model + threshold)', '#D4812B', 'white'),
    (6.5, 2.8,  5, 1.2, 'Prediction Log Store\n(PostgreSQL / CSV)',       '#8B4513', 'white'),
    (4.5, 0.8,  3, 1.2, 'Monitoring & Drift\n(Evidently AI)',             '#B03060', 'white'),
]

for x, y, w, h, txt, fc, tc in boxes:
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                          facecolor=fc, edgecolor='white', linewidth=1.5, alpha=0.88)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, txt, ha='center', va='center',
            color=tc, fontsize=8.5, fontweight='bold', multialignment='center')

arw = dict(arrowstyle='->', color='#333', lw=1.5)
ax.annotate('', xy=(6, 8.0), xytext=(6, 8.5),  arrowprops=arw)
ax.annotate('', xy=(6, 6.5), xytext=(6, 7.0),  arrowprops=arw)
ax.annotate('', xy=(2.5, 3.9), xytext=(3, 5.0), arrowprops=arw)
ax.annotate('', xy=(9, 3.9), xytext=(8.5, 5.0), arrowprops=arw)
ax.annotate('', xy=(6, 1.9), xytext=(9, 2.8),  arrowprops=arw)

plt.tight_layout()
arch_path = Path('Tumushiime_deployment_architecture.png')
plt.savefig(str(arch_path), dpi=120, bbox_inches='tight')
if matplotlib.get_backend().lower() != 'agg':
    plt.show()
plt.close(fig)
print(f'Architecture diagram saved: {arch_path.resolve()}')

SyntaxError: unterminated string literal (detected at line 16) (1414881776.py, line 16)

### Q3 - API Contract Example

**Endpoint:** `POST /predict`
**Framework:** FastAPI
**Description:** Accepts a single transaction record and returns the predicted return risk score.

**Input payload (JSON):**
```json
{
  "DescriptionClean": "white hanging heart glass decoration",
  "UnitPrice": 2.55,
  "AbsQuantity": 6,
  "InvoiceMonth": 11,
  "InvoiceWeekday": 3,
  "InvoiceHour": 14,
  "CustomerIDMissing": 0,
  "Country": "United Kingdom"
}
```

**Output response (JSON):**
```json
{
  "PredictedReturnFlag": 0,
  "ReturnProbability": 0.0312,
  "InferenceLabel": "Low Return Risk",
  "ModelName": "Gradient Boosting",
  "ThresholdUsed": 0.187
}
```

**Business rule:** operations teams configure their own alert threshold. The API always returns the raw probability so downstream systems can apply any cut-off appropriate to their cost tolerance.

## Q4. Ethical, Privacy, and Reliability Considerations

Deploying this return-risk system in production requires controls across ethics, privacy, and reliability:

1. **Ethical fairness and bias**  
   The model may over-flag certain countries, product groups, or customer segments because of historical patterns. This can lead to unfair treatment if used blindly. Bias checks should be run by subgroup, and model outputs should support human decision-making rather than fully automate punitive actions.

2. **Transparency and explainability**  
   Operations teams need to understand why a transaction is marked high-risk. The service should expose probability scores and key drivers (for example, major contributing features), and document the threshold logic so decisions are auditable.

3. **Privacy and data protection**  
   Transaction records may include personal or commercially sensitive information. Data minimization, encryption in transit/at rest, strict access controls, anonymized logs, and retention limits are needed to reduce misuse and regulatory risk.

4. **Reliability and model drift**  
   Customer behavior can change over time, causing concept drift and performance decay. Continuous monitoring (precision/recall, false-positive rate, calibration, drift metrics), scheduled retraining, and rollback to a previous stable artefact should be part of operations.

5. **Operational robustness and governance**  
   Real-world systems must handle bad inputs, outages, and dependency failures. Input validation, fallback/manual review workflows, API health checks, versioned artefacts, and audit logs improve resilience and accountability.

Overall, responsible deployment means balancing predictive value with fairness safeguards, privacy-by-design, and ongoing reliability monitoring.

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import joblib

# Simulated FastAPI /predict handler
def predict_endpoint(request, artefact, tfidf, le):
    """
    Simulates the FastAPI POST /predict handler.
    Accepts a single transaction record dict and returns a risk score dict.
    """
    num_feats = artefact['numeric_features']
    model     = artefact['model']
    threshold = artefact['threshold']

    country = request.get('Country', 'United Kingdom')
    if country not in le.classes_:
        country = 'United Kingdom'
    country_enc = le.transform([country])[0]

    row = {
        'UnitPrice':         request['UnitPrice'],
        'AbsQuantity':       request['AbsQuantity'],
        'LineValueAbs':      request['UnitPrice'] * request['AbsQuantity'],
        'InvoiceMonth':      request['InvoiceMonth'],
        'InvoiceWeekday':    request['InvoiceWeekday'],
        'InvoiceHour':       request['InvoiceHour'],
        'CustomerIDMissing': request['CustomerIDMissing'],
        'DescriptionLength': len(request.get('DescriptionClean', '')),
        'CountryEncoded':    country_enc,
    }

    X_num  = csr_matrix([[row[f] for f in num_feats]])
    X_text = tfidf.transform([request.get('DescriptionClean', '')])
    X_inf  = hstack([X_num, X_text])

    # HistGBM requires dense input
    from scipy.sparse import issparse as _issparse
    if 'HistGradient' in type(model).__name__ or 'Gradient Boosting' in artefact.get('model_name', ''):
        X_inf = X_inf.toarray() if _issparse(X_inf) else X_inf
    prob  = float(model.predict_proba(X_inf)[0, 1])
    flag  = int(prob >= threshold)
    label = 'High Return Risk' if flag == 1 else 'Low Return Risk'

    return {
        'PredictedReturnFlag': flag,
        'ReturnProbability':   round(prob, 4),
        'InferenceLabel':      label,
        'ModelName':           artefact['model_name'],
        'ThresholdUsed':       round(threshold, 4),
    }


# Run simulated inference
art_loaded = joblib.load('Tumushiime best model.joblib')

train_df   = pd.read_csv('Tumushiime.csv', usecols=['Country', 'DescriptionClean'])
le_demo    = LabelEncoder().fit(train_df['Country'])
tfidf_demo = TfidfVectorizer(max_features=200, ngram_range=(1, 2), min_df=5)
tfidf_demo.fit(train_df['DescriptionClean'].fillna(''))

sample_requests = [
    {'DescriptionClean': 'white hanging heart glass decoration',
     'UnitPrice': 2.55, 'AbsQuantity': 6, 'InvoiceMonth': 11,
     'InvoiceWeekday': 3, 'InvoiceHour': 14, 'CustomerIDMissing': 0,
     'Country': 'United Kingdom'},
    {'DescriptionClean': 'manual postage bank charges dotcom adjustment',
     'UnitPrice': 1.20, 'AbsQuantity': 18, 'InvoiceMonth': 12,
     'InvoiceWeekday': 1, 'InvoiceHour': 9, 'CustomerIDMissing': 1,
     'Country': 'United Kingdom'},
]

responses = []
for req in sample_requests:
    resp = predict_endpoint(req, art_loaded, tfidf_demo, le_demo)
    responses.append(resp)
    print("Request :")
    print(json.dumps(req, indent=2))
    print("Response:")
    print(json.dumps(resp, indent=2))
    print()

api_path = Path('Tumushiime_api_example_response.json')
with open(api_path, 'w') as f:
    json.dump({'requests': sample_requests, 'responses': responses}, f, indent=2)
print(f"API example saved: {api_path.resolve()}")